# Colab: train the single-subject baseline

Select a Colab T4 GPU kernel. Run the settings and training cells. The default `baseline` mode uses 40 epochs and the original batch size of 32; choose `smoke` for a brief check. Data, results and checkpoints under `/content` disappear when the server is removed. The original training program automatically evaluates the test set at the end.


In [1]:
print("Colab kernel is responding")

Colab kernel is responding


Prepare the colab project copy

In [2]:
from pathlib import Path
import subprocess
import zipfile

repo = Path("/content/emg2qwerty")
model_zip = Path("/content/cnn-bigru-colab.zip")

if not model_zip.is_file():
    raise FileNotFoundError(
        "Model ZIP not found in /content. Check where it was uploaded."
    )

if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/EdwinKam/emg2qwerty.git", str(repo)],
        check=True,
        env={**__import__("os").environ, "GIT_LFS_SKIP_SMUDGE": "1"},
    )

with zipfile.ZipFile(model_zip) as archive:
    for name in archive.namelist():
        target = repo / name
        if target.exists() and target.read_bytes() != archive.read(name):
            raise RuntimeError(f"Existing file differs; inspect it first: {target}")
    archive.extractall(repo)

print("CNN–BiGRU model files are available in Colab.")

CNN–BiGRU model files are available in Colab.


In [8]:
REPO_URL = "https://github.com/EdwinKam/emg2qwerty.git"
PROJECT_REF = "376923b"  # Published commit; use another published ref if needed.
MODEL = "cnn_bigru_ctc"  # Use "cnn_bigru_ctc" after publishing the model branch.
RUN_MODE = "baseline"  # "baseline": full 40 epochs; "smoke": short test
BASELINE_EPOCHS = 40
SMOKE_STEPS = 10
BATCH_SIZE = 32  # Original baseline default; smoke mode uses 4.


# ZIP in Google Drive; used only when the subject files are absent in this runtime.
DRIVE_ZIP = "/content/drive/MyDrive/ECE247A_final_project/89335547.zip"


In [9]:
import os, subprocess, sys, time, threading, signal, zipfile, shutil, tempfile
from datetime import datetime
from pathlib import Path
try:
    import google.colab
    import torch
except ImportError as exc:
    raise RuntimeError('Select a hosted Colab GPU kernel first.') from exc
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before continuing.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
# Keep the subprocess environment compatible with Colab and print complete errors.
os.environ['MPLBACKEND'] = 'Agg'
os.environ['HYDRA_FULL_ERROR'] = '1'
root = Path('/content')
repo = root / 'emg2qwerty'
data = root / 'emg2qwerty-subject-89335547'
archive = root / 'subject-89335547.zip'
env = root / 'emg2qwerty-env'
mamba = root / 'bin/micromamba'

status_path = root / 'emg2qwerty-status.json'
def status(stage, state='running', detail=''):
    record = {'stage': stage, 'state': state, 'detail': detail, 'updated': datetime.now().astimezone().isoformat()}
    temporary = status_path.with_suffix('.tmp')
    temporary.write_text(__import__('json').dumps(record, indent=2))
    temporary.replace(status_path)
    print(f"[{record['updated']}] {stage}: {state} {detail}", flush=True)
def run(args, stage='Setup', cwd=None, env=None, log_path=None):
    status(stage, detail='Starting: ' + ' '.join(map(str, args)))
    started = time.monotonic()
    process = subprocess.Popen([str(x) for x in args], cwd=cwd, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1,
                               start_new_session=True)
    done = threading.Event()
    def heartbeat():
        while not done.wait(30):
            print(f'[{stage}] Still running ({int(time.monotonic()-started)} seconds elapsed)...', flush=True)
    thread = threading.Thread(target=heartbeat, daemon=True)
    thread.start()
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            if log_path:
                with open(log_path, 'a') as log:
                    log.write(line)
        rc = process.wait()
        if rc:
            status(stage, 'failed', f'Exit code {rc}')
            raise subprocess.CalledProcessError(rc, args)
        status(stage, 'completed', f'{int(time.monotonic()-started)} seconds')
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        status(stage, 'interrupted', 'Subprocess stopped')
        raise
    finally:
        done.set()
git_env = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1')
if not repo.exists():
    run(['git', 'clone', REPO_URL, repo], env=git_env, stage='Clone repository')
if not (repo / '.git').exists():
    raise RuntimeError(f'{repo} is not a Git checkout; inspect it before continuing.')
run(['git', '-C', repo, 'fetch', 'origin'], env=git_env)
if subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain', '--untracked-files=no'], text=True).strip():
    raise RuntimeError(f'{repo} has local changes; preserve them before continuing.')
run(['git', '-C', repo, 'checkout', '--detach', PROJECT_REF], env=git_env)
print('Code commit:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip(), flush=True)
model_config = repo / 'config' / 'model' / f'{MODEL}.yaml'
if not model_config.is_file():
    raise RuntimeError(f'Model {MODEL!r} is not present at PROJECT_REF={PROJECT_REF!r}. Publish the model branch and set PROJECT_REF to it before running.')
if not mamba.exists():
    run(['bash', '-c', 'set -o pipefail; curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba'])
if not (env / 'bin/python').exists():
    run([mamba, 'create', '-y', '-p', env, '-c', 'conda-forge', 'python=3.10', 'pip=24.0'], stage='Create Python environment')
python = env / 'bin/python'
marker = env / '.dependencies-ready'
if not marker.exists():
    run([python, '-m', 'pip', 'install', '-r', repo / 'requirements.txt'], stage='Install dependencies')
    run([python, '-m', 'pip', 'install', '--no-deps', '-e', repo])
    run([python, '-c', 'import torch, torchaudio, pytorch_lightning, hydra, kenlm; assert torch.cuda.is_available(), "PyTorch cannot access GPU"'])
    marker.touch()
# Use the exact configured train/validation/test sessions for subject 89335547.
check = r"""import sys, yaml
from pathlib import Path
cfg = yaml.safe_load(Path(sys.argv[1]).read_text())
root = Path(sys.argv[2])
missing = [s['session'] + '.hdf5' for split in ('train', 'val', 'test') for s in cfg['dataset'][split] if not (root / (s['session'] + '.hdf5')).is_file()]
if missing:
 print('Missing', len(missing), 'configured session files. Examples:', *missing[:3], sep='\n  ', flush=True)
 sys.exit(1)
"""
def data_ready():
    result = subprocess.run([str(python), '-c', check, str(repo / 'config/user/single_user.yaml'), str(data)], capture_output=True, text=True)
    if result.stdout: print(result.stdout, end='', flush=True)
    if result.stderr: print(result.stderr, end='', flush=True)
    if result.returncode not in (0, 1):
        raise RuntimeError(f'Dataset check failed with exit code {result.returncode}; see details above.')
    return result.returncode == 0
if not data_ready():
    if not archive.exists():
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            print('Mount Google Drive and approve access when prompted.', flush=True)
            drive.mount('/content/drive')
        source = Path(DRIVE_ZIP)
        if not source.is_file():
            raise FileNotFoundError(f'Subject ZIP not found: {source}. Update DRIVE_ZIP in the settings cell.')
        status('Copy subject ZIP from Drive', detail=f'{source} ({source.stat().st_size / 1e9:.2f} GB)')
        copied, total, last_report = 0, source.stat().st_size, 0
        with source.open('rb') as src, tempfile.NamedTemporaryFile(dir=root, prefix='subject-copy-', suffix='.part', delete=False) as dest:
            temporary_path = Path(dest.name)
            try:
                while True:
                    chunk = src.read(1024 * 1024)
                    if not chunk: break
                    dest.write(chunk)
                    copied += len(chunk)
                    if time.monotonic() - last_report >= 10:
                        status('Copy subject ZIP from Drive', detail=f'{copied / 1e9:.2f}/{total / 1e9:.2f} GB ({100 * copied / total:.1f}%)')
                        last_report = time.monotonic()
            except BaseException:
                print(f'Copy interrupted. Partial file kept at {temporary_path}', flush=True)
                raise
        if copied != total or not zipfile.is_zipfile(temporary_path):
            raise RuntimeError(f'The Drive copy is incomplete or invalid: {temporary_path}')
        temporary_path.replace(archive)
        status('Copy subject ZIP from Drive', 'completed', f'{copied / 1e9:.2f} GB')
    if not zipfile.is_zipfile(archive):
        raise RuntimeError(f'{archive} is not a valid ZIP; inspect the file before retrying.')
    # Copy only session files into the dedicated data directory, without overwriting existing files.
    data.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        members = [m for m in zf.infolist() if not m.is_dir() and m.filename.lower().endswith('.hdf5')]
        if not members:
            raise RuntimeError('The ZIP contains no .hdf5 session files; inspect its contents.')
        names = [Path(m.filename).name for m in members]
        if len(names) != len(set(names)):
            raise RuntimeError('The ZIP contains duplicate HDF5 filenames; inspect it before extracting.')
        for index, member in enumerate(members, 1):
            dest = data / Path(member.filename).name
            if dest.exists():
                print(f'Preserving existing file: {dest}', flush=True)
                continue
            temp = dest.with_name(dest.name + '.part')
            if temp.exists():
                raise RuntimeError(f'Partial extraction exists at {temp}; inspect it before retrying.')
            status('Extract subject data', detail=f'{index}/{len(members)}: {dest.name}')
            with zf.open(member) as source, temp.open('xb') as target:
                shutil.copyfileobj(source, target, length=1024 * 1024)
            temp.replace(dest)
    if not data_ready():
        raise RuntimeError('Subject ZIP is missing configured sessions. Check the Box dataset and config/user/single_user.yaml.')
status('Dataset verification', 'completed', 'Required train, validation and test files found')
run_dir = root / 'emg2qwerty-results' / time.strftime('%Y%m%d-%H%M%S')
run_dir.mkdir(parents=True, exist_ok=False)
if RUN_MODE not in ('baseline', 'smoke'):
    raise ValueError("RUN_MODE must be 'baseline' or 'smoke'.")
command = [
    python, '-m', 'emg2qwerty.train',
    'user=single_user', f'model={MODEL}', f'dataset.root={data}',
    'trainer.accelerator=gpu', 'trainer.devices=1',
    'num_workers=2', f'hydra.run.dir={run_dir}',
]
if RUN_MODE == 'smoke':
    command += [
        'trainer.max_epochs=1', f'+trainer.max_steps={SMOKE_STEPS}',
        '+trainer.val_check_interval=5', '+trainer.limit_val_batches=2',
        '+trainer.limit_test_batches=1', 'lr_scheduler.scheduler.warmup_epochs=1',
        'batch_size=4',
    ]
    description = f'Smoke test: up to 1 epoch and {SMOKE_STEPS} steps'
else:
    # Original baseline training configuration: full data and validation.
    command += [f'trainer.max_epochs={BASELINE_EPOCHS}', f'batch_size={BATCH_SIZE}']
    description = f'Baseline: up to {BASELINE_EPOCHS} epochs, batch size {BATCH_SIZE}; full validation and test'
status('Training and evaluation', detail=f'{description}; results: {run_dir}')
run(command, cwd=repo, stage='Training and evaluation', log_path=run_dir / 'console.log')
import shutil
result_zip = shutil.make_archive(str(run_dir), 'zip', run_dir)
status('Results', 'completed', f'Archive: {result_zip}. Download before removing the server.')


GPU: NVIDIA L4
[2026-11-24T18:40:38.126064+00:00] Setup: running Starting: git -C /content/emg2qwerty fetch origin
[2026-11-24T18:40:38.705375+00:00] Setup: completed 0 seconds
[2026-11-24T18:40:38.710321+00:00] Setup: running Starting: git -C /content/emg2qwerty checkout --detach 376923b
HEAD is now at 376923b Update README.md
[2026-11-24T18:40:38.714375+00:00] Setup: completed 0 seconds
Code commit: 376923bcb4892d8cddae0ca7cd3d472ae11d93aa
[2026-11-24T18:40:38.771809+00:00] Dataset verification: completed Required train, validation and test files found
[2026-11-24T18:40:38.773443+00:00] Training and evaluation: running Baseline: up to 40 epochs, batch size 32; full validation and test; results: /content/emg2qwerty-results/20261124-184038
[2026-11-24T18:40:38.774292+00:00] Training and evaluation: running Starting: /content/emg2qwerty-env/bin/python -m emg2qwerty.train user=single_user model=cnn_bigru_ctc dataset.root=/content/emg2qwerty-subject-89335547 trainer.accelerator=gpu traine

In [7]:
from pathlib import Path

log = Path("/content/emg2qwerty-results/20261124-182410/console.log")
text = log.read_text(errors="replace")

print("=== MODEL ===")
if "CNNBiGRUCTCModule" in text:
    print("CNN–BiGRU")
elif "TDSConvCTCModule" in text:
    print("TDS baseline")
else:
    print("Could not identify the model from the log.")

print("\n=== CER AND SER ===")
lines = [
    line for line in text.splitlines()
    if any(metric in line for metric in [
        "val/CER", "val/SER", "test/CER", "test/SER"
    ])
]
print("\n".join(lines[-12:]) if lines else "No metric lines found.")

=== MODEL ===
CNN–BiGRU

=== CER AND SER ===
monitor_metric: val/CER
                                                                      Epoch 0, global step 5: 'val/CER' reached 707.18561 (best 707.18561), saving model to '/content/emg2qwerty-results/20261124-182410/checkpoints/epoch=0-step=5.ckpt' as top 1
                                                                      Epoch 0, global step 10: 'val/CER' reached 685.02997 (best 685.02997), saving model to '/content/emg2qwerty-results/20261124-182410/checkpoints/epoch=0-step=10.ckpt' as top 1
│          val/CER          │     685.0299682617188     │
│          val/SER          │     91.0179672241211      │
│         test/CER          │     631.7052001953125     │
│         test/SER          │     88.3293685913086      │
                  'val/CER': 685.0299682617188,
                  'val/SER': 91.0179672241211}],
                   'test/CER': 631.7052001953125,
                   'test/SER': 88.3293685913086}],


## Check runtime status independently

Run the next cell by itself on the **same Colab server** after the active cell has stopped. It only inspects files, logs, and processes; it does not resume training. A single notebook kernel runs one cell at a time, so a status cell submitted while training runs will wait in the queue.


In [10]:
# Read-only runtime status. Run this cell after stopping the active cell.
# A notebook kernel executes one cell at a time, so this cannot run concurrently
# with a training cell on the same kernel. It does not start or stop anything.
from pathlib import Path
import json, os, shutil, subprocess
root = Path('/content')
if not root.exists():
    raise RuntimeError('Select the same Colab runtime to inspect its files.')
print('=== Colab runtime status ===')
record = root / 'emg2qwerty-status.json'
if record.exists():
    print('Last recorded notebook status:', record.read_text())
    print('This is a last report, not proof the process is still running.')
else:
    print('No status record: this may be a run of the earlier notebook version.')
for path in [root / 'emg2qwerty', root / 'emg2qwerty-env/bin/python',
             root / 'emg2qwerty-env/.dependencies-ready',
             root / 'subject-89335547.zip.part', root / 'subject-89335547.zip', root / 'emg2qwerty-subject-89335547',
             root / 'emg2qwerty-data-2021-08.tar.gz.part',
             root / 'emg2qwerty-data-2021-08.tar.gz', root / 'emg2qwerty-data-2021-08',
             root / 'emg2qwerty/data']:
    if path.is_file():
        print(f'{path}: exists, {path.stat().st_size / 1e9:.3f} GB')
    elif path.is_dir():
        print(f'{path}: directory exists')
    elif path.is_symlink():
        print(f'{path}: broken link -> {os.readlink(path)}')
    else:
        print(f'{path}: not found')
data = root / 'emg2qwerty-subject-89335547'
if data.is_dir():
    files = list(data.rglob('*.hdf5'))
    print(f'Extracted HDF5 files found: {len(files)} (file count alone does not prove dataset is complete)')
    print('metadata.csv:', 'found' if (data / 'metadata.csv').exists() else 'not found')
print('Disk free: %.2f GB' % (shutil.disk_usage(root).free / 1e9))
print('\n=== Relevant running processes ===')
r = subprocess.run(['ps', '-eo', 'pid,etime,args'], text=True, capture_output=True)
lines = [line for line in r.stdout.splitlines() if any(x in line for x in ('emg2qwerty.train', 'pip install', 'micromamba create', 'emg2qwerty-data-2021-08.tar.gz')) and 'ps -eo' not in line]
print('\n'.join(lines) if lines else 'None found (does not guarantee a clean stop).')
print('\n=== Result files and recent training log ===')
results = root / 'emg2qwerty-results'
if results.exists():
    logs = sorted(results.rglob('console.log'), key=lambda p: p.stat().st_mtime, reverse=True)
    checkpoints = list(results.rglob('*.ckpt'))
    archives = list(results.glob('*.zip'))
    print('Checkpoints:', *[str(p) for p in checkpoints], sep='\n  ')
    print('Result ZIPs:', *[str(p) for p in archives], sep='\n  ')
    if logs:
        print('Latest log:', logs[0])
        print('\n'.join(logs[0].read_text(errors='replace').splitlines()[-30:]))
    else:
        print('No training log found; training may not have started.')
else:
    print('No results directory found; training may not have started.')


=== Colab runtime status ===
Last recorded notebook status: {
  "stage": "Results",
  "state": "completed",
  "detail": "Archive: /content/emg2qwerty-results/20261124-184038.zip. Download before removing the server.",
  "updated": "2026-11-24T19:12:19.274489+00:00"
}
This is a last report, not proof the process is still running.
/content/emg2qwerty: directory exists
/content/emg2qwerty-env/bin/python: exists, 0.017 GB
/content/emg2qwerty-env/.dependencies-ready: exists, 0.000 GB
/content/subject-89335547.zip.part: not found
/content/subject-89335547.zip: exists, 4.726 GB
/content/emg2qwerty-subject-89335547: directory exists
/content/emg2qwerty-data-2021-08.tar.gz.part: not found
/content/emg2qwerty-data-2021-08.tar.gz: not found
/content/emg2qwerty-data-2021-08: not found
/content/emg2qwerty/data: not found
Extracted HDF5 files found: 18 (file count alone does not prove dataset is complete)
metadata.csv: not found
Disk free: 229.08 GB

=== Relevant running processes ===
None found (do

## View the latest results

Run this cell to display the end of the latest training log and list its checkpoints. A smoke test verifies the pipeline, not final accuracy.


In [6]:
# Show metrics from the latest run. This does not train the model.
from pathlib import Path
root = Path('/content/emg2qwerty-results')
logs = sorted(root.glob('*/console.log'), key=lambda p: p.stat().st_mtime, reverse=True)
if not logs:
    print('No training log found on this server.')
else:
    log = logs[0]
    print('Run:', log.parent)
    print('\n'.join(log.read_text(errors='replace').splitlines()[-60:]))
    print('\nCheckpoints:')
    for path in sorted(log.parent.rglob('*.ckpt')):
        print(f'  {path.name} ({path.stat().st_size / 1e6:.1f} MB)')


Run: /content/emg2qwerty-results/20261124-182410

Validation: 0it [00:00, ?it/s]

Validation:   0%|          | 0/2 [00:00<?, ?it/s]

Validation DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 0:   0%|          | 0/1341 [00:01<?, ?it/s, loss=155, v_num=0]

                                                                      Epoch 0, global step 10: 'val/CER' reached 685.02997 (best 685.02997), saving model to '/content/emg2qwerty-results/20261124-182410/checkpoints/epoch=0-step=10.ckpt' as top 1

Epoch 0:   0%|          | 0/1341 [00:01<?, ?it/s, loss=155, v_num=0]`Trainer.fit` stopped: `max_steps=10` reached.

Epoch 0:   0%|          | 0/1341 [00:01<?, ?it/s, loss=155, v_num=0]LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: 0it [00:00, ?it/s]
Validation DataLoader 0: 100%|██████████| 2/2 [00:00<00:00, 19.99it/s]
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━

## Save results and stop the GPU

Download the ZIP under `/content/emg2qwerty-results` using the Colab Contents view, or copy it to Google Drive. Verify the results are saved, then select **Colab → Remove Server**. A fresh runtime downloads and extracts the dataset again.
